# Tool Integration Deep Dive

**Author:** Shuvam Banerji Seal

**Last Updated:** April 2026

## Learning Objectives

Master the art of building and integrating tools with AI agents:

1. Design tool schemas
2. Implement tool validation
3. Handle tool errors gracefully
4. Build custom tool libraries
5. Test and debug tools
6. Chain tools together

## What Makes a Good Tool?

### Characteristics

1. **Single Responsibility**
   - One tool = one clear action
   - Example: ❌ "ProcessDocument" vs ✅ "ExtractText" + "SummarizeText"

2. **Clear Interface**
   - Well-defined inputs and outputs
   - Comprehensive docstrings
   - Type hints

3. **Predictable Behavior**
   - Deterministic results
   - Clear error conditions
   - Documented edge cases

4. **Efficiency**
   - Fast execution
   - Low resource usage
   - Caching where appropriate

5. **Observability**
   - Logging of inputs/outputs
   - Performance metrics
   - Error tracking

## Tool Design Patterns

### 1. Simple Function Tool

```python
def add_numbers(a: int, b: int) -> int:
    """Add two numbers together.
    
    Args:
        a: First number
        b: Second number
    
    Returns:
        Sum of a and b
    """
    return a + b
```

### 2. Tool with Validation

```python
def divide_numbers(numerator: float, denominator: float) -> float:
    """Divide two numbers.
    
    Args:
        numerator: Number to divide
        denominator: Number to divide by
    
    Returns:
        Result of division
    
    Raises:
        ValueError: If denominator is zero
    """
    if denominator == 0:
        raise ValueError("Cannot divide by zero")
    return numerator / denominator
```

### 3. Tool with Error Handling

```python
def fetch_url(url: str, timeout: int = 10) -> str:
    """Fetch content from a URL.
    
    Args:
        url: URL to fetch
        timeout: Request timeout in seconds
    
    Returns:
        HTML content of the page
    """
    import requests
    
    try:
        response = requests.get(url, timeout=timeout)
        response.raise_for_status()
        return response.text
    except requests.RequestException as e:
        raise RuntimeError(f"Failed to fetch {url}: {str(e)}")
```

## Tool Schema Definition

Schemas tell agents what parameters a tool accepts.

### JSON Schema Format

```json
{
  "type": "object",
  "properties": {
    "query": {
      "type": "string",
      "description": "Search query"
    },
    "max_results": {
      "type": "integer",
      "description": "Maximum number of results",
      "default": 10
    }
  },
  "required": ["query"]
}
```

### Python Pydantic Version

```python
from pydantic import BaseModel, Field

class SearchInput(BaseModel):
    query: str = Field(
        description="Search query"
    )
    max_results: int = Field(
        default=10,
        description="Maximum number of results"
    )
```

## Tool Library Architecture

```
┌─────────────────────────────────────┐
│     Tool Library                    │
├─────────────────────────────────────┤
│                                     │
│  ┌──────────────────────────────┐  │
│  │  Base Tool Class             │  │
│  │  - validate_inputs()         │  │
│  │  - execute()                 │  │
│  │  - get_schema()              │  │
│  └──────────────────────────────┘  │
│           ▲                        │
│           │                        │
│  ┌────────┴─────────┬──────────┐   │
│  │                  │          │   │
│  ▼                  ▼          ▼   │
│ ┌───────┐    ┌──────────┐ ┌─────┐ │
│ │Search │    │Calculator│ │File │ │
│ │Tool   │    │Tool      │ │Tool │ │
│ └───────┘    └──────────┘ └─────┘ │
│                                     │
└─────────────────────────────────────┘
```

## Building a Tool Class

```python
from abc import ABC, abstractmethod
import logging
from typing import Any, Dict

class BaseTool(ABC):
    """Base class for all tools."""
    
    def __init__(self, name: str, description: str):
        self.name = name
        self.description = description
        self.logger = logging.getLogger(name)
    
    @abstractmethod
    def validate_inputs(self, inputs: Dict[str, Any]) -> bool:
        """Validate input parameters."""
        pass
    
    @abstractmethod
    def execute(self, **kwargs) -> str:
        """Execute the tool."""
        pass
    
    @abstractmethod
    def get_schema(self) -> Dict[str, Any]:
        """Return tool schema for LLM."""
        pass
    
    def call(self, **kwargs) -> str:
        """Safe execution wrapper."""
        try:
            if not self.validate_inputs(kwargs):
                return "Error: Invalid inputs"
            
            self.logger.info(f"Executing {self.name} with {kwargs}")
            result = self.execute(**kwargs)
            self.logger.info(f"Result: {result}")
            return result
        
        except Exception as e:
            self.logger.error(f"Execution failed: {e}")
            return f"Error: {str(e)}"


class SearchTool(BaseTool):
    """Search the web."""
    
    def __init__(self):
        super().__init__(
            name="search",
            description="Search the web for information"
        )
    
    def validate_inputs(self, inputs: Dict[str, Any]) -> bool:
        return "query" in inputs and len(inputs["query"]) > 0
    
    def execute(self, query: str, **kwargs) -> str:
        # Implement search logic
        return f"Search results for: {query}"
    
    def get_schema(self) -> Dict[str, Any]:
        return {
            "name": self.name,
            "description": self.description,
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Search query"
                    }
                },
                "required": ["query"]
            }
        }
```

## Error Handling Strategies

### 1. Input Validation

```python
def validate_email(email: str) -> bool:
    """Validate email format."""
    import re
    pattern = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'
    return re.match(pattern, email) is not None

def send_email(to: str, subject: str, body: str) -> str:
    if not validate_email(to):
        return f"Error: Invalid email address: {to}"
    # ... send email
```

### 2. Graceful Degradation

```python
def get_user_data(user_id: int) -> Dict:
    try:
        # Try primary source
        return fetch_from_primary_db(user_id)
    except Exception as e:
        logger.warning(f"Primary fetch failed: {e}")
        try:
            # Fall back to cache
            return fetch_from_cache(user_id)
        except Exception as cache_error:
            logger.error(f"Cache also failed: {cache_error}")
            # Return default/empty response
            return {"error": "Could not fetch user data"}
```

### 3. Retry with Backoff

```python
import asyncio
from typing import Callable, Any

async def retry_with_backoff(
    func: Callable,
    max_retries: int = 3,
    initial_delay: float = 1.0
) -> Any:
    """Retry function with exponential backoff."""
    delay = initial_delay
    
    for attempt in range(max_retries):
        try:
            return await func()
        except Exception as e:
            if attempt == max_retries - 1:
                raise
            
            logger.info(f"Attempt {attempt + 1} failed, retrying in {delay}s")
            await asyncio.sleep(delay)
            delay *= 2  # Exponential backoff
```

## Tool Composition and Chaining

### Sequential Chaining

```python
class ToolChain:
    def __init__(self, tools: List[BaseTool]):
        self.tools = {tool.name: tool for tool in tools}
    
    async def execute_chain(self, steps: List[Dict]) -> Any:
        """Execute tools in sequence.
        
        Args:
            steps: [
                {"tool": "search", "params": {"query": "..."}},
                {"tool": "analyze", "params": {"data": "<result>"}}  # <result> = output of previous
            ]
        """
        result = None
        
        for step in steps:
            tool_name = step["tool"]
            params = step["params"]
            
            # Replace <result> placeholder
            if "<result>" in str(params):
                params = replace_placeholders(params, {"result": result})
            
            tool = self.tools[tool_name]
            result = await tool.call(**params)
        
        return result
```

### Parallel Tool Execution

```python
async def execute_tools_parallel(
    tools: Dict[str, BaseTool],
    executions: List[Dict]
) -> Dict[str, Any]:
    """Execute multiple tools in parallel."""
    tasks = []
    
    for exec_spec in executions:
        tool_name = exec_spec["tool"]
        params = exec_spec["params"]
        tool = tools[tool_name]
        
        tasks.append(
            asyncio.create_task(tool.call(**params))
        )
    
    results = await asyncio.gather(*tasks)
    return {exec["name"]: result for exec, result in zip(executions, results)}
```

## Testing Tools

### Unit Tests

```python
import pytest
from unittest.mock import patch, MagicMock

class TestSearchTool:
    def setup_method(self):
        self.tool = SearchTool()
    
    def test_valid_query(self):
        result = self.tool.execute(query="Python AI agents")
        assert isinstance(result, str)
        assert len(result) > 0
    
    def test_empty_query(self):
        assert not self.tool.validate_inputs({"query": ""})
    
    @patch('requests.get')
    def test_network_error(self, mock_get):
        mock_get.side_effect = ConnectionError("Network failed")
        result = self.tool.call(query="test")
        assert "Error" in result
    
    def test_schema(self):
        schema = self.tool.get_schema()
        assert "parameters" in schema
        assert "query" in schema["parameters"]["properties"]
```

### Integration Tests

```python
def test_tool_chain_integration():
    chain = ToolChain([search_tool, analyze_tool])
    
    result = chain.execute_chain([
        {"tool": "search", "params": {"query": "AI agents"}},
        {"tool": "analyze", "params": {"data": "<result>"}}
    ])
    
    assert isinstance(result, dict)
    assert "summary" in result
```

## Debugging Tools

### Logging Best Practices

```python
import logging
import json
from datetime import datetime

class ToolLogger:
    def __init__(self, tool_name: str):
        self.logger = logging.getLogger(tool_name)
        handler = logging.FileHandler(f"{tool_name}.log")
        formatter = logging.Formatter(
            '%(asctime)s - %(name)s - %(levelname)s - %(message)s'
        )
        handler.setFormatter(formatter)
        self.logger.addHandler(handler)
    
    def log_execution(self, inputs: Dict, outputs: Dict, duration_ms: float):
        log_entry = {
            "timestamp": datetime.now().isoformat(),
            "inputs": inputs,
            "outputs": outputs,
            "duration_ms": duration_ms
        }
        self.logger.info(json.dumps(log_entry))
```

### Tracing Execution

```python
from functools import wraps
import time

def trace_execution(func):
    """Decorator to trace function execution."""
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"[TRACE] Calling {func.__name__}")
        print(f"[TRACE] Args: {args}")
        print(f"[TRACE] Kwargs: {kwargs}")
        
        start = time.time()
        try:
            result = func(*args, **kwargs)
            elapsed = time.time() - start
            print(f"[TRACE] Result: {result} (took {elapsed:.3f}s)")
            return result
        except Exception as e:
            elapsed = time.time() - start
            print(f"[TRACE] Error: {e} (took {elapsed:.3f}s)")
            raise
    
    return wrapper
```

## Real-World Tool Examples

### Example 1: Database Query Tool

```python
class DatabaseQueryTool(BaseTool):
    def __init__(self, connection_string: str):
        super().__init__("db_query", "Query database")
        self.conn_string = connection_string
    
    def execute(self, sql: str, max_rows: int = 10) -> str:
        import sqlite3
        
        conn = sqlite3.connect(self.conn_string)
        cursor = conn.cursor()
        
        # Prevent SQL injection
        if "DROP" in sql.upper() or "DELETE" in sql.upper():
            return "Error: Destructive queries not allowed"
        
        cursor.execute(sql)
        results = cursor.fetchmany(max_rows)
        conn.close()
        
        return json.dumps(results)
```

### Example 2: Email Tool

```python
class EmailTool(BaseTool):
    def __init__(self, smtp_config: Dict):
        super().__init__("send_email", "Send email")
        self.smtp_config = smtp_config
    
    def execute(self, to: str, subject: str, body: str) -> str:
        import smtplib
        from email.mime.text import MIMEText
        
        # Validate
        if not self.validate_email(to):
            return f"Error: Invalid email {to}"
        
        msg = MIMEText(body)
        msg["Subject"] = subject
        msg["From"] = self.smtp_config["from"]
        msg["To"] = to
        
        with smtplib.SMTP(**self.smtp_config) as server:
            server.send_message(msg)
        
        return f"Email sent to {to}"
```

## Best Practices

### 1. **Tool Design**
- Single responsibility principle
- Clear, concise docstrings
- Comprehensive type hints
- Sensible defaults

### 2. **Error Handling**
- Validate all inputs
- Handle network failures
- Implement retry logic
- Return structured errors

### 3. **Performance**
- Cache expensive operations
- Use async for I/O
- Monitor execution time
- Set reasonable timeouts

### 4. **Security**
- Validate and sanitize inputs
- Never expose secrets
- Implement rate limiting
- Log sensitive operations

### 5. **Testing**
- Unit test each tool
- Integration tests for chains
- Test error scenarios
- Mock external dependencies

## Exercise: Build Custom Tool Library

### Task
Create a tool library with:

1. **Base Tool Class** with validation and error handling
2. **Web Search Tool** that fetches and summarizes results
3. **Calculator Tool** with expression validation
4. **Data Processing Tool** for CSV/JSON
5. **Tool Chain** for sequential execution

### Requirements
- Comprehensive docstrings
- Input validation
- Error handling
- Logging
- Unit tests

## Key Takeaways

- **Good tools** have single responsibility and clear interfaces
- **Validation** prevents cascading failures
- **Error handling** is critical for production reliability
- **Testing** tools independently ensures robustness
- **Composition** of tools creates powerful workflows
- **Monitoring** and logging enable debugging
- **Security** must be considered from the start
- **Documentation** makes tools usable by agents